
# Product Label Prediction (5 Labels) — Alternative Vectorization Comparison

This notebook avoids TF-IDF and compares alternative vectorization strategies for the **text-level multi-label product prediction** task over:

- flower
- oil
- gummies
- vape
- topical

It includes:

1. robust data loading from the pair-level file  
2. conversion to a text-level 5-label dataset  
3. split by `text_id` to avoid leakage  
4. comparison of alternative vectorization methods:
   - CountVectorizer (word n-grams)
   - HashingVectorizer (word n-grams)
   - Character n-grams
   - Word2Vec averaged embeddings
5. model comparison under those vectorizations  
6. threshold tuning on the validation set  
7. final test evaluation  
8. error analysis  
9. saving outputs


In [ ]:

# =========================
# 0. Imports
# =========================
import os
import re
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss
)

from gensim.models import Word2Vec

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:

# =========================
# 1. Paths
# =========================
PROJECT_ROOT = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis")
PAIR_PATH = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis/data_processing/ data/ final/pair_level_gpt41mini_clean.csv")
OUTPUT_DIR = PROJECT_ROOT / "Modeling" / "product_label_prediction_outputs_altvec"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("PAIR_PATH    =", PAIR_PATH)
print("PAIR exists? =", PAIR_PATH.exists())
print("OUTPUT_DIR   =", OUTPUT_DIR)


In [ ]:

# =========================
# 2. Helpers
# =========================
TARGET_LABELS = ["flower", "oil", "gummies", "vape", "topical"]

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\n", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def clean_product_name(x):
    x = str(x).strip().lower()
    aliases = {
        "gummy": "gummies",
        "gummies": "gummies",
        "flower": "flower",
        "oil": "oil",
        "tincture": "oil",
        "vape": "vape",
        "topical": "topical",
    }
    return aliases.get(x, x)

def find_col(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(f"Could not find any of columns: {candidates}")
    return None

def evaluate_multilabel(y_true, y_pred, label_names, model_name):
    row = {
        "model": model_name,
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }
    per_label = pd.DataFrame({
        "label": label_names,
        "precision": precision_score(y_true, y_pred, average=None, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=None, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=None, zero_division=0),
    })
    return row, per_label

def tune_thresholds(y_true, prob_matrix, label_names, grid=None):
    if grid is None:
        grid = np.round(np.arange(0.10, 0.91, 0.05), 2)
    best_thresholds = []
    label_scores = []
    for j, label in enumerate(label_names):
        yt = y_true[:, j]
        probs = prob_matrix[:, j]
        best_t = 0.5
        best_f1 = -1
        for t in grid:
            yp = (probs >= t).astype(int)
            score = f1_score(yt, yp, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_t = float(t)
        best_thresholds.append(best_t)
        label_scores.append(best_f1)
    return np.array(best_thresholds), pd.DataFrame({
        "label": label_names,
        "best_threshold": best_thresholds,
        "best_val_f1": label_scores
    })

def apply_thresholds(prob_matrix, thresholds):
    return (prob_matrix >= thresholds.reshape(1, -1)).astype(int)

def decision_to_unit_interval(scores):
    return 1 / (1 + np.exp(-scores))

def tokenize_w2v(text):
    text = normalize_text(text).lower()
    return re.findall(r"[a-zA-Z][a-zA-Z0-9_\-]+", text)

def average_w2v(tokens, model, dim):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(dim, dtype=float)
    return np.mean(vecs, axis=0)


In [ ]:

# =========================
# 3. Load pair-level data
# =========================
pair_df = pd.read_csv(PAIR_PATH)
print(pair_df.shape)
print(pair_df.columns.tolist())
pair_df.head()


In [ ]:

# =========================
# 4. Identify columns and build text-level 5-label dataset
# =========================
text_col = find_col(pair_df, ["text", "full_text", "clean_text", "comment_text", "body", "content"])
text_id_col = find_col(pair_df, ["text_id", "id", "record_id", "doc_id"])
product_col = find_col(pair_df, ["product", "product_label", "product_type", "product_form"])

pair_work = pair_df[[text_id_col, text_col, product_col]].copy()
pair_work.columns = ["text_id", "text", "product"]

pair_work["text"] = pair_work["text"].map(normalize_text)
pair_work["product"] = pair_work["product"].map(clean_product_name)

pair_work = pair_work[pair_work["text"].str.len() > 0].copy()
pair_work = pair_work[pair_work["product"].isin(TARGET_LABELS)].copy()
pair_work = pair_work.drop_duplicates()

print(pair_work.shape)
pair_work.head()


In [ ]:

label_df = (
    pair_work.groupby("text_id")
    .agg({
        "text": "first",
        "product": lambda s: sorted(set(s))
    })
    .reset_index()
)

mlb = MultiLabelBinarizer(classes=TARGET_LABELS)
Y = mlb.fit_transform(label_df["product"])
Y_df = pd.DataFrame(Y, columns=TARGET_LABELS)

text_level_df = pd.concat([label_df[["text_id", "text"]], Y_df], axis=1)
text_level_df["label_count"] = text_level_df[TARGET_LABELS].sum(axis=1)
text_level_df["is_multilabel"] = (text_level_df["label_count"] > 1).astype(int)

print(text_level_df.shape)
text_level_df.head()


In [ ]:

text_level_path = OUTPUT_DIR / "text_level_5label_dataset.csv"
text_level_df.to_csv(text_level_path, index=False)
print("Saved:", text_level_path)


In [ ]:

# =========================
# 5. Train/validation/test split by text_id
# =========================
train_df, temp_df = train_test_split(
    text_level_df,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

X_train = train_df["text"].tolist()
X_val   = val_df["text"].tolist()
X_test  = test_df["text"].tolist()

y_train = train_df[TARGET_LABELS].values.astype(int)
y_val   = val_df[TARGET_LABELS].values.astype(int)
y_test  = test_df[TARGET_LABELS].values.astype(int)


In [ ]:

print("Training size:", len(X_train))
print("Blank texts in train:", sum((not isinstance(x, str)) or (not x.strip()) for x in X_train))

label_dist = pd.DataFrame({
    "train_pos": y_train.sum(axis=0),
    "val_pos": y_val.sum(axis=0),
    "test_pos": y_test.sum(axis=0),
}, index=TARGET_LABELS)
label_dist


In [ ]:

# =========================
# 6. Model helpers
# =========================
def get_prob_from_pipeline_ovr(pipe, X):
    clf = pipe.named_steps["clf"]
    vec_name = [k for k in pipe.named_steps.keys() if k != "clf"][0]
    Xv = pipe.named_steps[vec_name].transform(X)

    if hasattr(clf, "predict_proba"):
        probs = np.column_stack([
            est.predict_proba(Xv)[:, 1] for est in clf.estimators_
        ])
    else:
        scores = np.column_stack([
            est.decision_function(Xv) for est in clf.estimators_
        ])
        probs = decision_to_unit_interval(scores)
    return probs

def fit_eval_pipeline(name, pipeline, X_train, y_train, X_val, y_val, X_test, y_test, label_names):
    pipeline.fit(X_train, y_train)
    val_prob = get_prob_from_pipeline_ovr(pipeline, X_val)
    thresholds, thr_df = tune_thresholds(y_val, val_prob, label_names)
    test_prob = get_prob_from_pipeline_ovr(pipeline, X_test)
    test_pred = apply_thresholds(test_prob, thresholds)
    summary, per_label = evaluate_multilabel(y_test, test_pred, label_names, name)
    summary["threshold_mean"] = float(np.mean(thresholds))
    summary["threshold_min"] = float(np.min(thresholds))
    summary["threshold_max"] = float(np.max(thresholds))
    return {
        "name": name,
        "pipeline": pipeline,
        "val_prob": val_prob,
        "test_prob": test_prob,
        "thresholds": thresholds,
        "threshold_df": thr_df,
        "test_pred": test_pred,
        "summary": summary,
        "per_label": per_label
    }

def fit_eval_dense_ovr(name, estimator, X_train, y_train, X_val, y_val, X_test, y_test, label_names):
    estimator.fit(X_train, y_train)
    if hasattr(estimator, "predict_proba"):
        val_prob = estimator.predict_proba(X_val)
        test_prob = estimator.predict_proba(X_test)
    else:
        val_prob = decision_to_unit_interval(estimator.decision_function(X_val))
        test_prob = decision_to_unit_interval(estimator.decision_function(X_test))
    thresholds, thr_df = tune_thresholds(y_val, val_prob, label_names)
    test_pred = apply_thresholds(test_prob, thresholds)
    summary, per_label = evaluate_multilabel(y_test, test_pred, label_names, name)
    summary["threshold_mean"] = float(np.mean(thresholds))
    summary["threshold_min"] = float(np.min(thresholds))
    summary["threshold_max"] = float(np.max(thresholds))
    return {
        "name": name,
        "model": estimator,
        "val_prob": val_prob,
        "test_prob": test_prob,
        "thresholds": thresholds,
        "threshold_df": thr_df,
        "test_pred": test_pred,
        "summary": summary,
        "per_label": per_label
    }


In [ ]:

# =========================
# 7. Alternative vectorization models
# =========================
results = []

# 7A. CountVectorizer + Logistic Regression
count_logreg = Pipeline([
    ("vec", CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        min_df=2,
        max_df=0.98,
        ngram_range=(1, 2),
        binary=False
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="liblinear"
        )
    ))
])
results.append(fit_eval_pipeline(
    "CountVectorizer + OVR LogisticRegression",
    count_logreg, X_train, y_train, X_val, y_val, X_test, y_test, TARGET_LABELS
))

# 7B. CountVectorizer + calibrated LinearSVC
count_svc = Pipeline([
    ("vec", CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        min_df=2,
        max_df=0.98,
        ngram_range=(1, 2),
        binary=True
    )),
    ("clf", OneVsRestClassifier(
        CalibratedClassifierCV(
            estimator=LinearSVC(class_weight="balanced"),
            method="sigmoid",
            cv=3
        )
    ))
])
results.append(fit_eval_pipeline(
    "CountVectorizer + Calibrated LinearSVC",
    count_svc, X_train, y_train, X_val, y_val, X_test, y_test, TARGET_LABELS
))

# 7C. HashingVectorizer + Logistic Regression
hash_logreg = Pipeline([
    ("vec", HashingVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        ngram_range=(1, 2),
        n_features=2**18,
        alternate_sign=False,
        norm="l2"
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="liblinear"
        )
    ))
])
results.append(fit_eval_pipeline(
    "HashingVectorizer + OVR LogisticRegression",
    hash_logreg, X_train, y_train, X_val, y_val, X_test, y_test, TARGET_LABELS
))

# 7D. Character n-grams + Logistic Regression
char_logreg = Pipeline([
    ("vec", CountVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.99
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="liblinear"
        )
    ))
])
results.append(fit_eval_pipeline(
    "Character n-grams + OVR LogisticRegression",
    char_logreg, X_train, y_train, X_val, y_val, X_test, y_test, TARGET_LABELS
))


In [ ]:

# =========================
# 8. Word2Vec averaged embeddings models
# =========================
X_train_tok = [tokenize_w2v(t) for t in X_train]
X_val_tok   = [tokenize_w2v(t) for t in X_val]
X_test_tok  = [tokenize_w2v(t) for t in X_test]

W2V_DIM = 100
w2v = Word2Vec(
    sentences=X_train_tok,
    vector_size=W2V_DIM,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10,
    seed=SEED
)

X_train_w2v = np.vstack([average_w2v(toks, w2v, W2V_DIM) for toks in X_train_tok])
X_val_w2v   = np.vstack([average_w2v(toks, w2v, W2V_DIM) for toks in X_val_tok])
X_test_w2v  = np.vstack([average_w2v(toks, w2v, W2V_DIM) for toks in X_test_tok])

print(X_train_w2v.shape, X_val_w2v.shape, X_test_w2v.shape)

w2v_logreg = OneVsRestClassifier(
    LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="liblinear"
    )
)
results.append(fit_eval_dense_ovr(
    "Word2Vec(avg) + OVR LogisticRegression",
    w2v_logreg, X_train_w2v, y_train, X_val_w2v, y_val, X_test_w2v, y_test, TARGET_LABELS
))

w2v_svc = OneVsRestClassifier(
    CalibratedClassifierCV(
        estimator=LinearSVC(class_weight="balanced"),
        method="sigmoid",
        cv=3
    )
)
results.append(fit_eval_dense_ovr(
    "Word2Vec(avg) + Calibrated LinearSVC",
    w2v_svc, X_train_w2v, y_train, X_val_w2v, y_val, X_test_w2v, y_test, TARGET_LABELS
))


In [ ]:

# =========================
# 9. Compare models / vectorization methods
# =========================
summary_df = pd.DataFrame([r["summary"] for r in results]).sort_values(
    ["macro_f1", "micro_f1"], ascending=False
).reset_index(drop=True)

summary_df


In [ ]:

summary_path = OUTPUT_DIR / "vectorization_model_comparison.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)


In [ ]:

plt.figure(figsize=(11, 5))
plt.bar(summary_df["model"], summary_df["macro_f1"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Macro F1")
plt.title("Product Label Prediction: Alternative Vectorization / Model Comparison")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "macro_f1_comparison.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

best_name = summary_df.iloc[0]["model"]
best_result = next(r for r in results if r["name"] == best_name)

print("Best model:", best_name)
display(best_result["threshold_df"])
display(best_result["per_label"])


In [ ]:

# =========================
# 10. Threshold tables for all models
# =========================
all_thresholds = []
for r in results:
    tmp = r["threshold_df"].copy()
    tmp.insert(0, "model", r["name"])
    all_thresholds.append(tmp)

all_thresholds_df = pd.concat(all_thresholds, ignore_index=True)
all_thresholds_df.to_csv(OUTPUT_DIR / "all_model_thresholds.csv", index=False)
all_thresholds_df.head(15)


In [ ]:

# =========================
# 11. Error analysis for best model
# =========================
best_pred = best_result["test_pred"].copy()

test_out = test_df[["text_id", "text"] + TARGET_LABELS].copy().reset_index(drop=True)
for i, lab in enumerate(TARGET_LABELS):
    test_out[f"pred_{lab}"] = best_pred[:, i]

def label_list_from_row(row, cols):
    return [c for c in cols if row[c] == 1]

test_out["true_labels"] = test_out.apply(lambda r: label_list_from_row(r, TARGET_LABELS), axis=1)
test_out["pred_labels"] = test_out.apply(lambda r: label_list_from_row(r, [f"pred_{c}" for c in TARGET_LABELS]), axis=1)
test_out["pred_labels"] = test_out["pred_labels"].apply(lambda xs: [x.replace("pred_", "") for x in xs])
test_out["correct_exact"] = (test_out["true_labels"].astype(str) == test_out["pred_labels"].astype(str)).astype(int)

errors = test_out[test_out["correct_exact"] == 0].copy()
errors["n_true"] = errors["true_labels"].apply(len)
errors["n_pred"] = errors["pred_labels"].apply(len)

errors_path = OUTPUT_DIR / "best_model_errors.csv"
errors.to_csv(errors_path, index=False)
print("Saved:", errors_path)
errors[["text_id", "text", "true_labels", "pred_labels"]].head(20)


In [ ]:

error_rows = []
for lab in TARGET_LABELS:
    fn = ((test_out[lab] == 1) & (test_out[f"pred_{lab}"] == 0)).sum()
    fp = ((test_out[lab] == 0) & (test_out[f"pred_{lab}"] == 1)).sum()
    tp = ((test_out[lab] == 1) & (test_out[f"pred_{lab}"] == 1)).sum()
    tn = ((test_out[lab] == 0) & (test_out[f"pred_{lab}"] == 0)).sum()
    error_rows.append({"label": lab, "TP": tp, "FP": fp, "FN": fn, "TN": tn})

error_df = pd.DataFrame(error_rows)
error_df.to_csv(OUTPUT_DIR / "best_model_error_breakdown_by_label.csv", index=False)
error_df


In [ ]:

print(
    "Interpretation guide:\n"
    "- Compare vectorization methods mainly with macro F1 and per-label F1.\n"
    "- Threshold tuning was done on the validation set separately for each label.\n"
    "- The best model is the one at the top of summary_df.\n"
    "- If Count/Hashing/Char models beat Word2Vec, sparse lexical features are more useful for this task.\n"
)
